In [1]:
import os 
from dotenv import load_dotenv

load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

In [2]:
# Model 
from langchain_openai import ChatOpenAI

model = ChatOpenAI()
model

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000011B236D5D30>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000011B236D67B0>, root_client=<openai.OpenAI object at 0x0000011B2348EE40>, root_async_client=<openai.AsyncOpenAI object at 0x0000011B236D6510>, model_kwargs={}, openai_api_key=SecretStr('**********'))

In [3]:
from langchain_core.documents import Document

documents = [
    Document(page_content="Dogs are best friends", metadata={"source": "dog"}),
    Document(page_content="Cats are cool", metadata={"source": "cat"}),
    Document(page_content="fish are cool", metadata={"source": "fish"}),
]

In [4]:
# Embeddings
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

# Vector Store
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents, embeddings)

vectorstore


In [5]:
vectorstore.similarity_search("cat")

[Document(id='eb97da53-51d6-43db-b2ce-784423214d8f', metadata={'source': 'cat'}, page_content='Cats are cool'),
 Document(id='4f64c95b-bbb4-4331-b71e-3369ec341572', metadata={'source': 'fish'}, page_content='fish are cool'),
 Document(id='1de00328-e8dd-4851-9a66-11bda05542fc', metadata={'source': 'dog'}, page_content='Dogs are best friends')]

In [6]:
vectorstore.similarity_search_with_score("cat")

[(Document(id='eb97da53-51d6-43db-b2ce-784423214d8f', metadata={'source': 'cat'}, page_content='Cats are cool'),
  0.33848339319229126),
 (Document(id='4f64c95b-bbb4-4331-b71e-3369ec341572', metadata={'source': 'fish'}, page_content='fish are cool'),
  0.43850618600845337),
 (Document(id='1de00328-e8dd-4851-9a66-11bda05542fc', metadata={'source': 'dog'}, page_content='Dogs are best friends'),
  0.4466676414012909)]

In [7]:
# Async Query
await vectorstore.asimilarity_search("cat")

[Document(id='eb97da53-51d6-43db-b2ce-784423214d8f', metadata={'source': 'cat'}, page_content='Cats are cool'),
 Document(id='4f64c95b-bbb4-4331-b71e-3369ec341572', metadata={'source': 'fish'}, page_content='fish are cool'),
 Document(id='1de00328-e8dd-4851-9a66-11bda05542fc', metadata={'source': 'dog'}, page_content='Dogs are best friends')]

In [8]:
# Retriver


In [9]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriver = RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriver.batch(['cat', 'fish'])

[[Document(id='eb97da53-51d6-43db-b2ce-784423214d8f', metadata={'source': 'cat'}, page_content='Cats are cool')],
 [Document(id='4f64c95b-bbb4-4331-b71e-3369ec341572', metadata={'source': 'fish'}, page_content='fish are cool')]]

In [10]:
# Second technicque

retriver = vectorstore.as_retriever(
    search_type= "similarity",
    search_kwargs={"k": 1}
    
)

retriver.batch(['cat', 'fish'])

[[Document(id='eb97da53-51d6-43db-b2ce-784423214d8f', metadata={'source': 'cat'}, page_content='Cats are cool')],
 [Document(id='4f64c95b-bbb4-4331-b71e-3369ec341572', metadata={'source': 'fish'}, page_content='fish are cool')]]

In [11]:
# RAG
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer the question using following context:

{question}

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([('human', message)])

rag_chain = {"context": retriver, "question": RunnablePassthrough()} | prompt | model

response = rag_chain.invoke('tell me about cat')

print(response.content)



Cats are cool.
